# 1. Collecte des données GPS 
* Objectif : Obtenir les coordonnées GPS des 35 villes
* Comment : API Nominatim API
* Données à extraire : CSV (ID_ville, ville, latitude, longitude)

(bibliothèque python également disponible)

In [ ]:
#liste des 35 villes
villes =["Mont Saint Michel",
"St Malo",
"Bayeux",
"Le Havre",
"Rouen",
"Paris",
"Amiens",
"Lille",
"Strasbourg",
"Chateau du Haut Koenigsbourg",
"Colmar",
"Eguisheim",
"Besancon",
"Dijon",
"Annecy",
"Grenoble",
"Lyon",
"Gorges du Verdon",
"Bormes les Mimosas",
"Cassis",
"Marseille",
"Aix en Provence",
"Avignon",
"Uzes",
"Nimes",
"Aigues Mortes",
"Saintes Maries de la mer",
"Collioure",
"Carcassonne",
"Ariege",
"Toulouse",
"Montauban",
"Biarritz",
"Bayonne",
"La Rochelle"]

# liste pour stocker les résultats
coordonnées = []

In [ ]:
# importer les bibliothèques nécessaires
import requests # pour faire des requêtes HTTP
import time # pour ajouter un délai entre les requêtes
import uuid # pour générer des identifiants uniques
import pandas as pd # pour manipuler les données
import csv # pour lire et écrire des fichiers CSV

# itérer sur chaque ville pour obtenir les coordonnées
for ville in villes:
    params = {
        'q': ville , # q=paramètre de recherche
        'format': 'json', #format=json pour obtenir la réponse en JSON
        'limit': 1 #pour obtenir une seule réponse par ville (car plusieurs réponses possibles)
    } 

#sans headers, la requête peut être bloquée par le serveur = code 403
    response = requests.get("https://nominatim.openstreetmap.org/search", params=params, headers={'User-Agent': 'Mozilla/5.0'})
        
    if response.status_code == 200: #si requête réussie
        data = response.json() #convertir la réponse en JSON
        if data: #si des données sont trouvées
            # créer un dictionnaire avec les coordonnées et la ville
            coord = { 
                'id': str(uuid.uuid4()),  # Générer un identifiant unique
                'ville': ville, #nom de la ville
                'latitude': data[0]['lat'], #latitude de la ville
                'longitude': data[0]['lon'] #longitude de la ville
                }
            coordonnées.append(coord)
        else:
            print(f"Aucune donnée trouvée pour {ville}") # si aucune donnée n'est trouvée
    else:
        print(f"Erreur lors de la requête pour {ville} :", response.status_code) #code d'erreur de la requête

    time.sleep(1)

In [ ]:
#sauvegarder les résultats dans un fichier csv
df=pd.DataFrame(coordonnées)
df.to_csv("coordonnées_villes.csv", index=False, encoding='utf-8') #index=False, ne pas inclure l'index dans le fichier CSV
print("Les coordonnées des villes ont été enregistrées dans 'coordonnées_villes.csv'.")

Les coordonnées des villes ont été enregistrées dans 'coordonnées_villes.csv'.


In [ ]:
#ouvrir le fichier csv et afficher son contenu
with open('coordonnées_villes.csv', mode='r', encoding='utf-8') as file: # ouvrir le fichier en mode lecture
    reader = csv.reader(file)
    for row in reader:
        print(row)

['id', 'ville', 'latitude', 'longitude']
['0ed70527-1544-42b3-bd0d-621f44989283', 'Mont Saint Michel', '48.6359541', '-1.5114600']
['277397f3-7214-4738-bad2-4d6c0a9d707b', 'St Malo', '48.6495180', '-2.0260409']
['0e49462a-e8df-4a3f-b29e-d61199e4a6cb', 'Bayeux', '49.2764624', '-0.7024738']
['b740dfcb-50e2-4682-ab83-0a82633c98a5', 'Le Havre', '49.4938975', '0.1079732']
['b0e82e13-f6c7-4bca-ac89-57d394894738', 'Rouen', '49.4404591', '1.0939658']
['07a9e7e1-1aed-4b22-9c8f-7e44578eb8e9', 'Paris', '48.8534951', '2.3483915']
['3e73f537-a969-4156-8954-16594706ca55', 'Amiens', '49.8941708', '2.2956951']
['cdae81a9-c33e-4989-a7a8-5961e70fd174', 'Lille', '50.6365654', '3.0635282']
['d8838c51-f37f-49fd-a059-ade96703a797', 'Strasbourg', '48.5846140', '7.7507127']
['5c6fdd54-a558-4e52-ab27-027a8214c7af', 'Chateau du Haut Koenigsbourg', '48.2494107', '7.3443202']
['7dd7daf6-2949-4cce-88d8-535711c8d0b5', 'Colmar', '48.0777517', '7.3579641']
['e257cee3-bb2f-4f4d-b813-0a5d5f972ff2', 'Eguisheim', '48.044

# 2. Collecte des données météo (prévisions +7 jours)
* Objectif : Obtenir les prévisions météo des 35 villes
* Comment : 
    * OpenWeatherMap - One Call API 
    * Critères : températures, pluie, humidité, qualité de l'air
* Sortie : 

In [27]:
#clé API OpenWeatherMap
API_KEY = 'cc056e880361f33e9906834f2e2a2148'

#test pour vérifier si la clé API est valide
response = requests.get(f'http://api.openweathermap.org/data/3.0/onecall?lat=48.8588443&lon=2.2943506&appid={API_KEY}&exclude=minutely,hourly,alerts')

print(response.status_code) # afficher le code de statut de la réponse

401


In [28]:

#créer les colonnes météo que je veux ajouter à mon fichier CSV
df['summary'] = None #summary
df['temp_jour'] = None #daily.temp.day
df['ressenti'] = None #daily.feels_like.day
df['humidite'] = None #daily.humidity
df['pluie_7j'] = None #daily.rain
df['prob_pluie'] = None #daily.pop
df['indice_uv'] = None #daily.uvi

print(df.columns)

Index(['id', 'ville', 'latitude', 'longitude', 'summary', 'temp_jour',
       'ressenti', 'humidite', 'pluie_7j', 'prob_pluie', 'indice_uv',
       'ville_affichable'],
      dtype='object')


In [ ]:
for i, row in df.iterrows():
    lat = row['latitude']
    lon = row['longitude']

    #paramètres pour la requête API OpenWeatherMap
    params = {
        'lat': lat,
        'lon': lon,
        'exclude': 'minutely,hourly,alerts', # Exclure les données non nécessaires
        'units': 'metric', # Unités métriques
        'appid': API_KEY # Clé API OpenWeatherMap
    }

    response = requests.get("https://api.openweathermap.org/data/2.5/onecall", params=params, headers={'User-Agent': 'Mozilla/5.0'})

    if response.status_code == 200:
        data = response.json()

        # Données sur 7 jours
        daily = data.get('daily', [])

        if daily:
            # Moyenne température sur 7 jours
            moy_temp = sum(jour['temp']['day'] for jour in daily[:7]) / 7
            humidite = sum(jour['humidity'] for jour in daily[:7]) / 7
            pluie = sum(jour.get('rain', 0) for jour in daily[:7])  # parfois pas de clé 'rain'

            df.at[i, 'temp_moyenne'] = round(moy_temp, 1)
            df.at[i, 'humidite'] = round(humidite, 1)
            df.at[i, 'pluie_7j'] = round(pluie, 1)

            print(f"Météo ajoutée pour {row['ville']}")
        else:
            print(f"Pas de données météo pour {row['ville']}")
    else:
        print(f"Erreur API pour {row['ville']}: {response.status_code}")

    time.sleep(1)  # Pause pour respecter les quotas de l’API

# Enregistrement du fichier enrichi
df.to_csv("villes_meteo.csv", index=False)
print(" Fichier 'villes_meteo.csv' exporté avec météo.")


Erreur API pour Mont Saint Michel: 401
Erreur API pour St Malo: 401
Erreur API pour Bayeux: 401
Erreur API pour Le Havre: 401


KeyboardInterrupt: 

# 3. Scraping Booking.com
* Quoi : récupérer les informations d'hôtels pour chaque ville
* Comment : scrapy
* Données à extraire : CSV (ID_hôtel, city_ID, nom_hôtel, url, latitude, longitude, note, description)

In [ ]:
from bs4 import BeautifulSoup

villes = df['ville'].tolist()  # Récupérer la liste des villes depuis le DataFrame
hotels_data = []

for ville in villes:
    url = f"https://www.booking.com/searchresults.fr.html?ss={villes.index(ville)}"]"
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
    
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
        hotels = soup.select('div', class_='sr_property_block_main_row')[:10]

        for hotel in hotels:
            name = hotel.find('span', class_='sr-hotel__name').get_text(strip=True)
            link = hotel.find('a', class_='hotel_name_link')['href'] if hotel.find('a', class_='hotel_name_link') else 'N/A'
            lat =
            long =
            rating = hotel.find('div', class_='bui-review-score__badge').get_text(strip=True) if hotel.find('div', class_='bui-review-score__badge') else 'N/A'
            description = hotel.find('div', class_='bui-text').get_text(strip=True) if hotel.find('div', class_='bui-text') else 'N/A'
            price = hotel.find('div', class_='bui-price-display__value').get_text(strip=True) if hotel.find('div', class_='bui-price-display__value') else 'N/A'

            hotels_data.append({
                'ville': ville,
                'name': name,
                'link': link,
                'latitude': lat,
                'longitude': long,
                'rating': rating,
                'description': description,
                'price': price
            })

time.sleep(2)  # Pause pour éviter de surcharger le serveur

SyntaxError: invalid syntax (3593846749.py, line 17)

# 4. Création du Data Lake (S3)
* Quoi : Stocker tous les fichiers CSV dans des buckets S3
* Contenu à stocker : Données météo par ville / Liste des villes avec GPS / Données des hôtels
* Nom du bucket S3 : kayak-data-lake

# 5. ETL vers un entrepôt SQL
* Outils : AWS RDS (MySQL ou PostgreSQL)
* Étapes : Créer des tables SQL pour cities, weather, hotels
* Charger les fichiers CSV depuis S3 dans la base de données avec un script ETL (Python, Airflow ou AWS Glue)
* Vérifier que les données sont bien normalisées (clé étrangère entre weather et cities, entre hotels et cities)

# 6. Visualisations
Outil recommandé : Plotly
Cartes à produire :
* Top 5 des villes avec la meilleure météo (selon ton indice)
* Top 20 hôtels (note utilisateur + météo favorable)

Représentation : cartes interactives avec clusters ou bulles



# Livrables finaux
* CSV enrichi → Stocké sur S3
* Base SQL sur AWS RDS contenant toutes les données
* Deux cartes Plotly :
    * Top 5 des destinations
    * Top 20 hôtels
* (Optionnel) : Documentation sur les critères météo utilisés + scripts utilisés